# Data Analysis

In this analysis phase, we will focus on computing the following key metrics for each product:

### 1. Volume (Sales Count)
- Defined as the **number of purchase events** associated with a product ID.
- Since there is no quantity column, each row represents a single purchase event.
- Multiple purchases of the same product by the same user appear as multiple rows.
- **Calculation:** count of purchase-event rows per product.

---

### 2. Revenue (Gross Income)
- Defined as the **sum of the price column** for purchase events.
- No data is available for shipping costs, wholesale costs, or taxes.
- This metric therefore represents **gross revenue**, not profit.
- **Calculation:** sum of prices per product.

---

### 3. Popularity (Unique Buyers)
- Defined as the **number of unique users** who purchased a product.
- Repeated purchases by the same user are counted once.
- Serves as a proxy for product popularity rather than purchase intensity.
- **Calculation:** count of distinct user IDs per product.

---

### 4. Change in Performance (Temporal Comparison)
- Measures how product performance evolves over time.
- Metrics are calculated **separately for November and December**.
- Enables comparison of volume, revenue, and popularity between the two periods.
- **Calculation:** per-metric comparison across months.

---

### 5. Conversion Rate
- Defined as the **percentage of cart events that result in a purchase**.
- Reflects how effectively cart additions convert into completed purchases.
- Product views are not considered due to missing data.
- **Calculation:**  
$$
\text{Conversion Rate} =
\frac{\text{Number of purchase events}}
{\text{Number of cart events}} \times 100
$$

---

These metrics together provide a structured view of **sales performance, customer reach, temporal trends, and conversion efficiency** in the absence of detailed quantity or browsing data.


In [7]:
# import libraries
import pandas as pd
import numpy as np

In [8]:
# Load the cleaned data
events = pd.read_parquet('./datasets/events.parquet.gz')

In [9]:
# Make a copy to separate purchase and cart events
purchases = events[events.event_type == 'purchase'].copy()
carts = events[events.event_type == 'cart'].copy()

In [10]:
# Creating month-specific columns for November and December

purchases.assign(
    november_count=np.where(purchases['event_time'].dt.month == 11, 1, 0),
    november_revenue=np.where(purchases['event_time'].dt.month == 11, purchases['price'], 0),
    november_user_id =np.where(purchases['event_time'].dt.month == 11, purchases['user_id'], None),
    december_count=np.where(purchases['event_time'].dt.month == 12, 1, 0),
    december_revenue=np.where(purchases['event_time'].dt.month == 12, purchases['price'], 0),
    december_user_id =np.where(purchases['event_time'].dt.month == 12, purchases['user_id'], None)
)

,event_time,event_type,product_id,brand,subcategory,category,product_identifier,user_id,user_session,price,november_count,november_revenue,november_user_id,december_count,december_revenue,december_user_id
1,2019-11-01 00:00:41+00:00,purchase,13200605,None,furniture.bedroom.bed,furniture,None,559368633,d6034fa2-41fb-4ac0-9051-55ea9fc9147a,566.30,1,566.30,559368633,0,0.00,None
2,2019-11-01 00:01:04+00:00,purchase,1005161,xiaomi,electronics.smartphone,electronics,1005161-xiaomi electronics.smartphone,513351129,e6b7ce9b-1938-4e20-976c-8b4163aea11d,211.92,1,211.92,513351129,0,0.00,None
5,2019-11-01 00:04:51+00:00,purchase,1004856,samsung,electronics.smartphone,electronics,1004856-samsung electronics.smartphone,562958505,0f039697-fedc-40fa-8830-39c1a024351d,128.42,1,128.42,562958505,0,0.00,None
7,2019-11-01 00:06:33+00:00,purchase,1801881,samsung,appliances.personal.massager,appliances,1801881-samsung appliances.personal.massager,557746614,4d76d6d3-fff5-4880-8327-e9e57b618e0e,488.80,1,488.80,557746614,0,0.00,None
8,2019-11-01 00:06:34+00:00,purchase,5800823,nakamichi,electronics.audio.subwoofer,electronics,5800823-nakamichi electronics.audio.subwoofer,514166940,8ef5214a-86ad-4d0b-8df3-4280dd411b47,123.56,1,123.56,514166940,0,0.00,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7023545,2019-12-31 17:01:33+00:00,purchase,1005105,apple,electronics.smartphone,electronics,1005105-apple electronics.smartphone,558444704,ca84a221-d0f7-4e5b-a01a-704fe4059b40,1275.81,0,0.00,None,1,1275.81,558444704
7023551,2019-12-31 17:01:36+00:00,purchase,4804055,apple,sport.bicycle,sport,4804055-apple sport.bicycle,572536740,89528571-8c10-4989-8efc-78e2c8590136,197.28,0,0.00,None,1,197.28,572536740
7023556,2019-12-31 17:01:38+00:00,purchase,1005126,apple,electronics.smartphone,electronics,1005126-apple electronics.smartphone,514945482,1714bc55-8528-4936-a678-c8385905e096,1266.94,0,0.00,None,1,1266.94,514945482
7023557,2019-12-31 17:01:38+00:00,purchase,5100816,xiaomi,apparel.shoes,apparel,5100816-xiaomi apparel.shoes,559909071,d0c877f1-fb93-4f6e-9ecc-5b7d124b08ba,29.95,0,0.00,None,1,29.95,559909071


In [20]:
# Temporary table for purchases

purchases_table = (
    purchases
    .assign(
        november_count=np.where(purchases['event_time'].dt.month == 11, 1, 0),
        november_revenue=np.where(purchases['event_time'].dt.month == 11, purchases['price'], 0),
        november_user_id =np.where(purchases['event_time'].dt.month == 11, purchases['user_id'], None),
        december_count=np.where(purchases['event_time'].dt.month == 12, 1, 0),
        december_revenue=np.where(purchases['event_time'].dt.month == 12, purchases['price'], 0),
        december_user_id =np.where(purchases['event_time'].dt.month == 12, purchases['user_id'], None),
    )
.groupby(['product_id', 'product_identifier'])\
.agg(
    november_volume = ('november_count', 'sum'),
    november_revenue = ('november_revenue', 'sum'),
    november_users = ('november_user_id', 'nunique'),
    december_volume = ('december_count', 'sum'),
    december_revenue = ('december_revenue', 'sum'),
    december_users = ('december_user_id', 'nunique')
)
    .assign(
        volume_diff = lambda x: x['december_volume'] - x['november_volume'],
        revenue_diff = lambda x: x['december_revenue'] - x['november_revenue'],
        user_diff = lambda x: x['december_users'] - x['november_users']
    ) .reset_index()
)

purchases_table.head()

,product_id,product_identifier,november_volume,november_revenue,november_users,december_volume,december_revenue,december_users,volume_diff,revenue_diff,user_diff
0,1000978,1000978-samsung electronics.smartphone,20,6135.32,17,16,4260.40,16,-4,-1874.92,-1
1,1001588,1001588-meizu electronics.smartphone,6,766.29,5,13,1652.55,12,7,886.26,7
2,1001605,1001605-apple electronics.smartphone,0,0.00,0,18,9806.76,16,18,9806.76,16
3,1001606,1001606-apple electronics.smartphone,0,0.00,0,11,5662.69,10,11,5662.69,10
4,1001618,1001618-apple electronics.smartphone,36,18059.76,25,7,4745.84,7,-29,-13313.92,-18
